<a href="https://colab.research.google.com/github/Derrickdc02/DIS-Project-Lensed-Galaxy/blob/main/notebooks/Figure2OOD_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Figure 2 OOD Reproduction (Colab)

Run `src/figure2.py` on Colab: the Adam et al. 2022 Figure 2 out-of-distribution
experiment. The ground-truth source is a "7" (not a galaxy); for each noise
level we lens it, add noise, and draw ONE posterior sample. At low noise the
likelihood dominates and the sample recovers the "7"; as the noise grows the
galaxy prior takes over and the sample looks like a galaxy.

**Runtime: GPU required** (Runtime > Change runtime type). Each of the 5 noise
levels is a single 8000-step posterior draw, checkpointed to Drive as it
finishes -- if Colab disconnects, just re-run the sampling cell and completed
levels are skipped.

Requirements on Drive: the trained checkpoint `probes_final/latest.pt`.
If you still have partial results from CSD3 (`figure2.parts/level_*.pt`),
copy them into the Drive parts dir below and those levels are skipped too.

## 1. Install + clone the repo (`local` branch)

In [ ]:
!pip install -q score-models==0.5.11 caustics==1.6.0
# If pip just downgraded numpy: Runtime > Restart session, then re-run from here.

REPO = '/content/DIS-Project-Lensed-Galaxy'
!git clone -q -b local https://github.com/Derrickdc02/DIS-Project-Lensed-Galaxy.git {REPO} \
    || git -C {REPO} pull -q

## 2. Mount Drive + paths

In [ ]:
from pathlib import Path
import torch

from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive/DIS-Project-Lensed-Galaxy')
CKPT  = DRIVE / 'probes_final' / 'latest.pt'          # trained score-model checkpoint
OUT   = DRIVE / 'probes_final' / 'figure2' / 'figure2.png'
PARTS = OUT.with_suffix('.parts')                     # per-noise-level checkpoints
OUT.parent.mkdir(parents=True, exist_ok=True)

assert CKPT.exists(), f'Checkpoint not found: {CKPT}'
assert torch.cuda.is_available(), 'No GPU -- switch the runtime type first'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'ckpt: {CKPT}')
print(f'out:  {OUT}')

## 3. Resume hygiene

`figure2.py` skips any `level_k.pt` already in the parts dir. A session that
died *mid-save* can leave a corrupt part behind, which would crash the resume,
so drop any part that fails to load.

In [ ]:
if PARTS.exists():
    for f in sorted(PARTS.glob('level_*.pt')):
        try:
            torch.load(f, map_location='cpu', weights_only=False)
            print(f'{f.name}: ok (will be skipped)')
        except Exception as e:
            print(f'{f.name}: corrupt ({e}) -- deleting')
            f.unlink()
else:
    print('No parts yet -- starting fresh')

## 4. Run

One 8000-step draw per noise level, sequential. Re-run this cell after any
disconnect; finished levels are skipped. For a quick smoke test drop STEPS to
500 first (delete the parts dir before the real run afterwards).

In [ ]:
STEPS  = 8000
NOISES = '0.001,0.1,0.8,1.0,5.0'
SEED   = 21

%cd {REPO}
!python src/figure2.py --ckpt "{CKPT}" --out "{OUT}" --steps {STEPS} --noises {NOISES} --seed {SEED}

## 5. Result

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(OUT), width=1100))
print(f'Figure saved to {OUT}')